## Weather Prediciton using Neural Network
# Introduction
blah

In [2]:
import numpy as np
import pandas as pd
import torch 
import torch.nn as nn
import openmeteo_requests, requests_cache, retry_requests
import time

device = "cuda" if torch.cuda.is_available() else "mps"
torch.manual_seed(42)

print(f"Using device {device}")

Using device mps


# Importing the dataset

In [3]:
cache = requests_cache.CachedSession(".cache") 
retry_session = retry_requests.retry(cache, retries=5, backoff_factor=5)
openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://archive-api.open-meteo.com/v1/archive"
params_to_import = {
    "latitude": 51.76,
    "longitude" : 18.08,
    "start_date" : "2000-01-01",
    "end_date": "2026-06-01",
    "hourly" : ["temperature_2m", "rain", "precipitation", "relative_humidity_2m", "dew_point_2m", "wind_speed_10m", "wind_direction_10m", "wind_speed_100m", "wind_direction_100m"]
}
responses = openmeteo.weather_api(url, params=params_to_import) 

response = responses[0]
print(f"Coordinates: {response.Latitude()} {response.Longitude()}")

hourly = response.Hourly()
temperature_2m = hourly.Variables(0).ValuesAsNumpy()
rain = hourly.Variables(1).ValuesAsNumpy()
precipitation = hourly.Variables(2).ValuesAsNumpy()
relative_humidity = hourly.Variables(3).ValuesAsNumpy()
dew_point = hourly.Variables(4).ValuesAsNumpy()
wind_speed_10m = hourly.Variables(5).ValuesAsNumpy()
wind_direction_10m = hourly.Variables(6).ValuesAsNumpy()
wind_speed_100m = hourly.Variables(7).ValuesAsNumpy()
wind_direction_100m = hourly.Variables(8).ValuesAsNumpy()

data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

data["temperature_2m"] = temperature_2m
data["rain"] = rain
data["precipitation"] = precipitation
data["relative_humidity_2m"] = relative_humidity
data["dew_point_2m"] = dew_point
data["wind_speed_10m"] = wind_speed_10m
data["wind_direction_10m"] = wind_direction_10m
data["wind_speed_100m"] = wind_speed_100m
data["wind_direction_100m"] = wind_direction_100m

hourly_dataframe = pd.DataFrame(data=data)
print("\nHourly data\n", hourly_dataframe)

Coordinates: 51.77504348754883 18.065692901611328

Hourly data
                             date  temperature_2m  rain  precipitation  \
0      2000-01-01 00:00:00+00:00         -2.1025   0.0            0.0   
1      2000-01-01 01:00:00+00:00         -2.4025   0.0            0.0   
2      2000-01-01 02:00:00+00:00         -2.8525   0.0            0.0   
3      2000-01-01 03:00:00+00:00         -3.0525   0.0            0.0   
4      2000-01-01 04:00:00+00:00         -3.4025   0.0            0.0   
...                          ...             ...   ...            ...   
231571 2026-06-01 19:00:00+00:00         16.0000   0.0            0.0   
231572 2026-06-01 20:00:00+00:00         15.1000   0.0            0.0   
231573 2026-06-01 21:00:00+00:00         14.4500   0.0            0.0   
231574 2026-06-01 22:00:00+00:00         13.9500   0.0            0.0   
231575 2026-06-01 23:00:00+00:00         13.3500   0.0            0.0   

        relative_humidity_2m  dew_point_2m  wind_speed_10m 